# 🧠 02-机器学习 · 推导笔记 10：神经网络与反向传播（手写全流程）

> 神经网络是深度学习的“地基”，**反向传播（Backpropagation）**是训练它的引擎。面试高频：手推两层网络的反向传播、解释链式法则、说清梯度消失。

> 💡 一句话类比：神经网络像一条**流水线评审**——原料（特征）先过一层“小评审”（神经元）给出初步结论，再过第二层“大评审”综合意见。训练就是：**发现最终结论错了，把责任按“谁贡献大谁多担”层层往回追**，逐一调整每个评审的判据。

**运行环境**：Python 3.8+，仅依赖 numpy（对照 cell 需要 sklearn）。

## 📖 本章导读

> **这一章讲什么**（全书第 9 章）：从感知机到多层感知机（万能逼近）；核心是**反向传播**——链式法则的三行账（输出层 $\delta$ → 隐层传播 → 参数梯度），最后讲梯度消失与激活函数进化。

**学完本章，你应该能**：
1. 手推输出层 $\delta^L=(a-y)\odot\sigma'(z^L)$ 与隐层 $\delta^l=(W^{l+1\top}\delta^{l+1})\odot\sigma'(z^l)$；
2. 手推参数梯度 $\partial L/\partial W^l=\delta^l a^{l-1\top}$；
3. 解释为什么深层网络用 sigmoid 会梯度消失、ReLU 为什么好；
4. 讲清 XOR 为什么单层感知机学不会、隐层在做什么。

> 📌 全书第 9 章，手推必考题。阅读路径：本章导读 → 前向 · BP 推导 · 激活 → 自测清单 → 本章小结。

## §1 从感知机到多层网络：为什么必须“深”一层

**感知机（单层）**：$y = \operatorname{sign}(w^{\top} x + b)$——只能画一条直线，**学不了 XOR**（异或不可线性分，见图：XOR 的 4 个点在平面上永远找不到一条直线把两类分开）。

**单层 ≠ 多层的关键**：

- 单层（无隐层）输出是输入的线性组合 → 表达能力等价于线性模型；
- 加入**非线性激活函数**的隐层后，网络变成“线性组合 + 非线性 + 线性组合……”——**万能逼近定理**：足够宽的含一个隐层的网络可逼近任意连续函数。

> 💡 通俗例子：XOR 就是“一个班的同学要么都高冷要么都活泼才合得来（同号→1）”。单条规则（只看身高）分不开；但先问“你是不是高个？”和“你是不是外向？”，再综合两个答案，就能分开了——这就是隐层在做的事。

两层神经网络：前向与反向传播

> 📊 图 4：两层网络（2-4-1）——蓝实线=前向传播，红虚线=反向传播；δ 公式见 §3 推导。

## §2 前向传播（两层网络，记熟记号）

输入 $x \in \mathbb{R}^d$，隐层 $h \in \mathbb{R}^m$，输出 $\hat{y} \in \mathbb{R}$（二分类/回归）。记 $W_1 \in \mathbb{R}^{m \times d}$、$b_1 \in \mathbb{R}^m$、$W_2 \in \mathbb{R}^{1 \times m}$、$b_2 \in \mathbb{R}$：

$$z_1 = W_1 x + b_1, \qquad h = \sigma(z_1)$$

$$z_2 = W_2 h + b_2, \qquad \hat{y} = \sigma(z_2)$$

损失用平方误差（除以 2 是为求导方便）：

$$L = \frac{1}{2}(y - \hat{y})^2$$

**激活函数及导数**（必须会背）：

| 激活 | 公式 | 导数 | 特点 |
|------|------|------|------|
| sigmoid | $\sigma(z) = \frac{1}{1+e^{-z}}$ | $\sigma(z)(1-\sigma(z))$ | 输出 (0,1)，易饱和 |
| tanh | $\tanh z = \frac{e^z - e^{-z}}{e^z + e^{-z}}$ | $1 - \tanh^2 z$ | 输出 (-1,1)，零中心 |
| ReLU | $\max(0, z)$ | $\mathbb{1}[z>0]$ | 不饱和，深度网络首选 |

## §3 反向传播推导（面试手推核心）

目标：算出 $\partial L / \partial W_2$、$\partial L / \partial W_1$。核心工具是**链式法则**：从输出层往回逐层算“误差信号”。

**第 1 步：输出层误差信号 $\delta_2$**

$$\delta_2 = \frac{\partial L}{\partial z_2} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z_2} = (\hat{y} - y) \cdot \sigma'(z_2)$$

于是

$$\frac{\partial L}{\partial W_2} = \delta_2 \cdot h^{\top}, \qquad \frac{\partial L}{\partial b_2} = \delta_2$$

**第 2 步：隐层误差信号 $\delta_1$（关键：责任回传）**

$$\delta_1 = \frac{\partial L}{\partial z_1} = \left( \frac{\partial L}{\partial z_2} \cdot \frac{\partial z_2}{\partial h} \right) \cdot \frac{\partial h}{\partial z_1} = \left( \delta_2 \cdot W_2^{\top} \right) \odot \sigma'(z_1)$$

（$\odot$ 是逐元素乘。）于是

$$\frac{\partial L}{\partial W_1} = \delta_1 \cdot x^{\top}, \qquad \frac{\partial L}{\partial b_1} = \delta_1$$

**统一视角（面试加分）**：对任意层 $l$，

$$\delta_l = \left( W_{l+1}^{\top} \delta_{l+1} \right) \odot \sigma'(z_l), \qquad \frac{\partial L}{\partial W_l} = \delta_l \, a_{l-1}^{\top}$$

这就是**误差反向传播**：输出层的误差 $\delta_2$ 通过 $W_2^{\top}$ “分发”回隐层，再乘以激活导数做“闸门”。

> 💡 直觉：隐层神经元“该负多少责”？= 它下游所有神经元对它的“抱怨”（$W^{\top}\delta$）之和，再乘上它自己当时“反应有多灵敏”（$\sigma'$，饱和时≈0，梯度就传不动——这就是**梯度消失**的源头）。

In [ ]:
import numpy as np

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

class MLP:
    """手写两层 MLP（sigmoid 隐层 + sigmoid 输出），全批量梯度下降"""
    def __init__(self, d, m, seed=0):
        rng = np.random.default_rng(seed)
        # 小随机初始化（对称破坏 + 避免饱和）
        self.W1 = rng.normal(0, 0.5, (m, d))
        self.b1 = np.zeros(m)
        self.W2 = rng.normal(0, 0.5, (1, m))
        self.b2 = np.zeros(1)

    def forward(self, X):
        self.z1 = X @ self.W1.T + self.b1
        self.h = sigmoid(self.z1)
        self.z2 = self.h @ self.W2.T + self.b2
        self.y_hat = sigmoid(self.z2).ravel()
        return self.y_hat

    def backward(self, X, y):
        n = len(X)
        delta2 = (self.y_hat - y) * (self.y_hat * (1 - self.y_hat))  # 输出层误差
        delta1 = (delta2.reshape(-1, 1) * self.W2) * (self.h * (1 - self.h))  # 隐层误差
        gW2 = delta2 @ self.h
        gb2 = delta2.sum()
        gW1 = delta1.T @ X
        gb1 = delta1.sum(axis=0)
        return gW1, gb1, gW2, gb2

    def train(self, X, y, epochs=3000, lr=1.0, verbose=True):
        losses = []
        for it in range(epochs):
            self.forward(X)
            loss = 0.5 * np.mean((y - self.y_hat) ** 2)
            losses.append(loss)
            gW1, gb1, gW2, gb2 = self.backward(X, y)
            self.W1 -= lr * gW1; self.b1 -= lr * gb1
            self.W2 -= lr * gW2; self.b2 -= lr * gb2
            if verbose and (it % 500 == 0 or it == epochs - 1):
                print(f"epoch {it:4d}   loss = {loss:.5f}")
        return losses

    def predict(self, X, threshold=0.5):
        return (self.forward(X) >= threshold).astype(int)

In [ ]:
# ---- 经典考题：用两层 MLP 学 XOR ----
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0], dtype=float)

net = MLP(d=2, m=4, seed=42)
net.train(X_xor, y_xor, epochs=5000, lr=1.5)

pred = net.predict(X_xor)
print("\nXOR 结果：")
for xx, yy, pp in zip(X_xor, y_xor, pred):
    print(f"  {xx} -> 预测 {pp} (真实 {int(yy)})  {'✅' if pp == yy else '❌'}")
print(f"训练集准确率 = {np.mean(pred == y_xor):.3f}  (应为 1.0)")

In [ ]:
# ---- sklearn 对照：手写 MLP vs MLPClassifier（同数据、同规模）----
from sklearn.neural_network import MLPClassifier

mlp_sk = MLPClassifier(
    hidden_layer_sizes=(4,), activation="logistic",
    solver="lbfgs", max_iter=1000, random_state=42,
)
mlp_sk.fit(X_xor, y_xor)
print(f"sklearn MLP 在 XOR 上准确率 = {mlp_sk.score(X_xor, y_xor):.3f}")

# 换个更有挑战的数据：二维月牙（非线性分类）
from sklearn.datasets import make_moons
Xm, ym = make_moons(n_samples=300, noise=0.15, random_state=0)
net2 = MLP(d=2, m=16, seed=1)
net2.train(Xm, ym, epochs=2000, lr=1.0, verbose=False)
print(f"手写 MLP 在月牙数据上准确率 = {np.mean(net2.predict(Xm) == ym):.3f}")

mlp_sk2 = MLPClassifier(hidden_layer_sizes=(16,), activation="logistic",
                        solver="lbfgs", max_iter=1000, random_state=1)
mlp_sk2.fit(Xm, ym)
print(f"sklearn MLP 在月牙数据上准确率 = {mlp_sk2.score(Xm, ym):.3f}")

## §4 工程要点（面试追问区）

**1. 参数初始化**：全零初始化会让所有神经元对称、更新相同 → 网络退化。用小的随机数（如 $\mathcal{N}(0, 0.01)$）或 Xavier/He 初始化（按 fan_in/fan_out 缩放方差）。

**2. 学习率**：太大 → 震荡/发散；太小 → 收敛极慢。实践中用学习率衰减或自适应优化器（Adam/Momentum）。

**3. 梯度消失/爆炸**：

- 深层时，$\delta_l = (W_{l+1}^{\top}\delta_{l+1}) \odot \sigma'(z_l)$ 层层相乘：sigmoid 导数最大 0.25，深度一高梯度指数衰减 → 浅层学不动（**梯度消失**）；
- 权重初始化太大 → 梯度指数爆炸；
- 缓解：ReLU 族激活、残差连接、BN、合理的初始化、梯度裁剪。

**4. 过拟合**：早停（验证集 loss 不再降就停）、L2 正则（权重衰减）、Dropout（训练时随机丢弃部分神经元，测试时全开并缩放）、数据增强。

**5. 为什么神经网络“能吃高维数据”而 KNN 不行**：参数化模型把信息压缩进权重，预测时只需 $O(参数量)$ 计算，不依赖存储全部训练样本；代价是训练要大量数据与算力。

## §5 常见 bug 与边界（面试踩坑区）

- **sigmoid 输出配 MSE + 分类**：训练慢（输出饱和区导数≈0）。分类一般用**交叉熵**（输出层配 softmax），回归才常用 MSE；
- **忘记对输入做缩放**：特征量纲差 1000 倍，梯度被大特征主导；
- **输出层误用 ReLU**：输出可能为负/无界，二分类应 sigmoid、多分类 softmax；
- **batch 太小导致 loss 毛刺**：正常现象，看趋势别盯单点；
- **梯度检查没做**：手写 BP 最容易在维度/转置上出错——用数值梯度 `(L(w+eps)-L(w-eps))/(2eps)` 对比验证（§3 手推完建议写一个）；
- **只报告训练准确率**：XOR 只有 4 个样本，训练准 100% 不代表泛化，要上独立测试集。

## §6 自测清单（L1 必会 / L2 进阶 / L3 挑战）

### L1 必会（8 问）

- [ ] 1. 口述前向传播两步（隐层→输出）公式
- [ ] 2. 手推 $\delta_2$、$\delta_1$、$\partial L/\partial W_2$、$\partial L/\partial W_1$
- [ ] 3. sigmoid/tanh/ReLU 的导数
- [ ] 4. 为什么单层感知机学不了 XOR
- [ ] 5. 链式法则在 BP 里用在哪里
- [ ] 6. 什么是梯度消失，为什么 sigmoid 容易触发
- [ ] 7. 全零初始化为什么不行
- [ ] 8. 输出层激活函数的选择（二分类/多分类/回归）

### L2 进阶（6 问）

- [ ] 9. 万能逼近定理的直观理解
- [ ] 10. 为什么分类用交叉熵而不用 MSE
- [ ] 11. 早停 / Dropout / 权重衰减的机制
- [ ] 12. 数值梯度检查怎么写
- [ ] 13. 梯度爆炸的成因与缓解
- [ ] 14. 学习率太大/太小各有什么表现

### L3 挑战（4 问）

- [ ] 15. 手写三层网络 BP（含 ReLU 隐层）
- [ ] 16. 用向量化 numpy 实现 mini-batch 训练
- [ ] 17. 推导交叉熵 + softmax 的梯度（$\hat{y} - y$ 的优美形式）
- [ ] 18. 解释 Xavier/He 初始化为什么按 fan_in/fan_out 缩放

### 自测要点

- 15 分钟内盲写两层 MLP 的前向 + 反向
- 能现场手推 $\delta_1 = (W_2^{\top}\delta_2) \odot \sigma'(z_1)$
- 能解释“深度网络为什么难训练”的三条原因

## 附录：参考与下节预告

- 《机器学习》周志华 第 5 章（神经网络、BP 推导）
- CS231n 课程笔记（反向传播、初始化、优化）
- 《Deep Learning》Goodfellow 第 6 章

### 下节预告：推导笔记 11 — 隐马尔可夫模型 HMM（前向/后向 / Viterbi / Baum-Welch）

> 💡 本笔记验收：`02-机器学习/README.md`「推导笔记」神经网络篇打勾。

## 🏁 本章小结

- **原理一句话**：MLP = 线性变换 + 非线性激活层层加工；BP = **链式法则**把输出误差分摊回每一层。
- **必会公式**：$\delta^L=(a-y)\odot\sigma'(z^L)$；$\delta^l=(W^{l+1\top}\delta^{l+1})\odot\sigma'(z^l)$；$\frac{\partial L}{\partial W^l}=\delta^l a^{l-1\top}$。
- **面试怎么考**：BP 三行推导；为什么深层难训（梯度消失）；sigmoid vs ReLU；XOR 为什么单层学不会（线性不可分）。
- **易错点**：$\delta$ 与激活 $a$ 逐元素相乘（维度要匹配）；偏置梯度就是 $\delta$ 本身。

> 自测清单没把握的条目，回到对应小节重读后再打勾。